# NewsQA RAG - Phase 2 End-to-End Baseline (Kaggle)
One locked RAG baseline on the 281 deduplicated resolved development questions. Retrieval uses BGE-M3 learned sparse retrieval and the BGE-large reranker; generation uses Gemini 3.1 Flash-Lite and RAGAS judging uses GLM-5.3-Flash through Fireworks.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, time, zipfile
def sha256_file(path,block_size=1024*1024):
    digest=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda:handle.read(block_size),b''): digest.update(block)
    return digest.hexdigest()
REPO_URL='https://github.com/ThomasdeCarpio/Text-Mining---NewsQA-RAG.git'
REPO_COMMIT='061894d2c5da4f6713b16908c59025cc332dbd3a'
HF_ARTIFACT_REPO_ID='ThomasAnderson2009/newsqa-rag-phase2-locked-v2'
HF_ARTIFACT_REVISION='locked-bge-m3-512-64-deduplicated-v2'
HF_ARTIFACT_FILENAME='artifacts/locked-bge-m3-512-64-deduplicated-v2/locked-bge-m3-512-64-deduplicated-v2.zip'
HF_ARTIFACT_SHA256='fc5d67b7acf6e8be0205ce00b8069b3b6c8dcce853f8671f2feb3887b2707a24'
RUN_MODE='smoke'  # Keep 'smoke' for the 5-question validation; change to 'full' only after review.
assert RUN_MODE in {'smoke','full'}
LOCKED_CHUNK_SIZE=512
LOCKED_CHUNK_OVERLAP=64
GENERATOR_MODEL='gemini-3.1-flash-lite'
GENERATOR_REASONING_EFFORT='minimal'
JUDGE_MODEL='accounts/fireworks/models/glm-5p3-flash'
JUDGE_REASONING_EFFORT='low'
JUDGE_MAX_TOKENS=2048
FIREWORKS_INPUT_PER_MILLION_USD=0.15
FIREWORKS_OUTPUT_PER_MILLION_USD=0.50
RERANKER_MODEL='BAAI/bge-reranker-large'
DEVELOPMENT_ARTICLES=50
TOP_K=20
RERANK_TOP_N=5
GENERATOR_MAX_TOKENS=512
GENERATOR_MIN_INTERVAL_SECONDS=4.2  # Conservative pacing for a 15 RPM free project.
JUDGE_PILOT_QUESTIONS=5 if RUN_MODE=='smoke' else 25
JUDGE_BATCH_SIZE=1
JUDGE_MAX_WORKERS=1
MAX_ATTEMPTS=5
EVAL_QUESTIONS=5 if RUN_MODE=='smoke' else None
EXPERIMENT_ID='phase2-e2e-baseline-dedup-v2-reasoning-smoke' if RUN_MODE=='smoke' else 'phase2-e2e-baseline-dedup-v2-reasoning'
CHECKPOINT_NAME=f'phase2_dedup_v2_reasoning_{RUN_MODE}_resume_checkpoint.zip'
AUTO_RESTORE_FROM_INPUT=True
LOCKED_INDEX_BUNDLE_NAME='locked-bge-m3-512-64-deduplicated-v2.zip'
LOCKED_ARTIFACT_DIRNAME='locked-bge-m3-512-64-deduplicated-v2'
KAGGLE_WORKING=Path('/kaggle/working')
KAGGLE_INPUT=Path('/kaggle/input')
PROJECT_ROOT=KAGGLE_WORKING/'Text-Mining---NewsQA-RAG'
WORK_ROOT=KAGGLE_WORKING/'newsqa_phase2'
DATA_ROOT=WORK_ROOT/'data'
INDEX_ROOT=WORK_ROOT/'index'
EXPERIMENTS=WORK_ROOT/'experiments'
SPECS=WORK_ROOT/'specs'
CACHE=WORK_ROOT/'retrieval_cache'
RESULTS=WORK_ROOT/'results'/RUN_MODE
LOGS=WORK_ROOT/'logs'/RUN_MODE


## 1. Kaggle setup
Enable Internet and one GPU. The notebook defaults to `RUN_MODE='smoke'`, which performs the complete pipeline on exactly five questions and exports reviewable results. Add and enable `GEMINI_API_KEY_1` for generation and `FIREWORKS_API_KEY` for judging. `HF_TOKEN` is optional because the pinned artifact is public. The reviewed deduplicated `512/64` corpus and BGE-M3 index are downloaded from an immutable artifact tag and verified by SHA-256; the notebook never rebuilds them from raw NewsQA. The held-out final-test partition is not used.

In [ ]:
from kaggle_secrets import UserSecretsClient
secrets=UserSecretsClient()
def optional_secret(name):
    try: return secrets.get_secret(name) or ''
    except Exception: return ''
os.environ['HF_TOKEN']=optional_secret('HF_TOKEN')
GENERATOR_API_KEY=optional_secret('GEMINI_API_KEY_1')
JUDGE_API_KEY=optional_secret('FIREWORKS_API_KEY')
assert GENERATOR_API_KEY, 'Add and enable GEMINI_API_KEY_1 for free-tier generation'
assert JUDGE_API_KEY, 'Add and enable FIREWORKS_API_KEY for RAGAS judging'
assert not REPO_COMMIT.startswith('SET_TO_'), 'Pin REPO_COMMIT after committing the Phase 2 implementation'
os.environ['HF_HOME']=str(KAGGLE_WORKING/'hf_cache')
os.environ.update({'TOKENIZERS_PARALLELISM':'false','OMP_NUM_THREADS':'1','MKL_NUM_THREADS':'1','PYTHONUNBUFFERED':'1','CUDA_VISIBLE_DEVICES':'0','LANGCHAIN_TRACING_V2':'false','LANGSMITH_TRACING':'false'})
if AUTO_RESTORE_FROM_INPUT and not WORK_ROOT.exists():
    checkpoints=sorted(KAGGLE_INPUT.rglob(CHECKPOINT_NAME),key=lambda path:path.stat().st_mtime,reverse=True)
    if checkpoints:
        WORK_ROOT.mkdir(parents=True,exist_ok=True); shutil.unpack_archive(checkpoints[0],WORK_ROOT); print('Restored:',checkpoints[0],flush=True)
for path in [DATA_ROOT,INDEX_ROOT,EXPERIMENTS,SPECS,CACHE,RESULTS,LOGS]: path.mkdir(parents=True,exist_ok=True)
if not PROJECT_ROOT.exists(): subprocess.run(['git','clone','--filter=blob:none',REPO_URL,str(PROJECT_ROOT)],check=True)
subprocess.run(['git','fetch','--depth=1','origin',REPO_COMMIT],cwd=PROJECT_ROOT,check=True,timeout=180)
subprocess.run(['git','checkout','--detach',REPO_COMMIT],cwd=PROJECT_ROOT,check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt'],cwd=PROJECT_ROOT,check=True)
from huggingface_hub import hf_hub_download
LOCKED_ARTIFACT_ROOT=DATA_ROOT/LOCKED_ARTIFACT_DIRNAME
locked_manifest_path=LOCKED_ARTIFACT_ROOT/'bundle_manifest.json'
if locked_manifest_path.exists():
    print('Using previously extracted locked artifact:',locked_manifest_path,flush=True)
else:
    locked_bundle=hf_hub_download(repo_id=HF_ARTIFACT_REPO_ID,repo_type='dataset',revision=HF_ARTIFACT_REVISION,filename=HF_ARTIFACT_FILENAME,token=os.environ['HF_TOKEN'] or None)
    assert Path(locked_bundle).name==LOCKED_INDEX_BUNDLE_NAME
    assert sha256_file(locked_bundle)==HF_ARTIFACT_SHA256, 'Downloaded artifact SHA-256 mismatch'
    LOCKED_ARTIFACT_ROOT.mkdir(parents=True,exist_ok=True); shutil.unpack_archive(locked_bundle,LOCKED_ARTIFACT_ROOT); print('Downloaded locked retrieval artifact:',locked_bundle,flush=True)
import pandas as pd, torch, yaml
from IPython.display import display
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator'
print('GPU:',torch.cuda.get_device_name(0),round(torch.cuda.get_device_properties(0).total_memory/2**30,1),'GiB')
print('Pinned commit:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=PROJECT_ROOT,text=True).strip())
print('Run mode:',RUN_MODE,'| questions:',EVAL_QUESTIONS or 281)


In [ ]:
def write_checkpoint():
    target=KAGGLE_WORKING/CHECKPOINT_NAME
    temporary=target.with_suffix('.zip.tmp')
    with zipfile.ZipFile(temporary,'w',compression=zipfile.ZIP_DEFLATED) as archive:
        for name in ['experiments','results','specs','retrieval_cache','logs']:
            root=WORK_ROOT/name
            if root.exists():
                for path in root.rglob('*'):
                    if path.is_file(): archive.write(path,path.relative_to(WORK_ROOT))
    temporary.replace(target); print('Resume checkpoint:',target,round(target.stat().st_size/2**20,1),'MiB',flush=True); return target
def sha256_file(path,block_size=1024*1024):
    digest=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda:handle.read(block_size),b''): digest.update(block)
    return digest.hexdigest()
def validate_and_prepare_locked_index():
    manifest_path=LOCKED_ARTIFACT_ROOT/'bundle_manifest.json'
    assert manifest_path.exists(), 'Locked artifact bundle_manifest.json is missing'
    bundle=json.loads(manifest_path.read_text())
    pipeline=bundle['pipeline']; chunking=pipeline['chunking']; sparse=pipeline['retrieval']['sparse']; reranker=pipeline['retrieval']['reranker']; stats=bundle['statistics']
    assert bundle['artifact_version']==HF_ARTIFACT_REVISION
    assert chunking['strategy']=='recursive' and chunking['chunk_size']==LOCKED_CHUNK_SIZE and chunking['chunk_overlap']==LOCKED_CHUNK_OVERLAP
    assert sparse['method']=='bge-m3' and sparse['model']=='BAAI/bge-m3'
    assert reranker['model']==RERANKER_MODEL and reranker['top_n']==RERANK_TOP_N
    assert stats['chunks']==22766 and stats['resolved_questions']==1152 and stats['development_questions']==281 and stats['final_test_questions']==871
    for relative,record in bundle['artifacts'].items():
        path=LOCKED_ARTIFACT_ROOT/relative; assert path.exists(), path
        assert path.stat().st_size==record['bytes'], f'Wrong size: {relative}'
        assert sha256_file(path)==record['sha256'], f'Wrong SHA-256: {relative}'
    chunks=LOCKED_ARTIFACT_ROOT/'chunks.jsonl'; testset=LOCKED_ARTIFACT_ROOT/'testset_resolved.jsonl'; sparse_index=LOCKED_ARTIFACT_ROOT/'bge_m3_sparse.pkl'
    config=yaml.safe_load((PROJECT_ROOT/'configs/config.yaml').read_text())
    config['chunking'].update({'strategy':'recursive','chunk_size':LOCKED_CHUNK_SIZE,'chunk_overlap':LOCKED_CHUNK_OVERLAP})
    config['llm'].update({'model':GENERATOR_MODEL,'temperature':0.0,'max_tokens':GENERATOR_MAX_TOKENS,'reasoning_effort':GENERATOR_REASONING_EFFORT})
    config['retrieval'].update({'retriever':'sparse','top_k':TOP_K})
    config['retrieval']['sparse'].update({'method':'bge-m3','model':'BAAI/bge-m3','device':'cuda'})
    config['retrieval']['reranker'].update({'enabled':True,'type':'cross-encoder','model':RERANKER_MODEL,'top_n':RERANK_TOP_N,'batch_size':8,'device':'cuda'})
    config_path=INDEX_ROOT/'phase2_locked_config.yaml'; config_path.write_text(yaml.safe_dump(config,sort_keys=False),encoding='utf-8')
    profile=json.loads((LOCKED_ARTIFACT_ROOT/'deduplication/deduplicated.variant.json').read_text())
    config_hash=hashlib.sha256(json.dumps(config,ensure_ascii=False,sort_keys=True,separators=(',',':')).encode()).hexdigest()
    profile['pipeline'].update({'config_path':str(config_path),'config_sha256':config_hash})
    profile['database'].update({'indexed':False,'chunk_count':stats['chunks']})
    profile['artifacts']['chunks']={'path':str(chunks),'bytes':chunks.stat().st_size,'sha256':sha256_file(chunks)}
    profile['artifacts']['testset_resolved']={'path':str(testset),'bytes':testset.stat().st_size,'sha256':sha256_file(testset)}
    profile['artifacts']['bm25']={'path':str(sparse_index),'bytes':sparse_index.stat().st_size,'sha256':sha256_file(sparse_index)}
    profile_path=INDEX_ROOT/'phase2_locked_variant.json'; profile_path.write_text(json.dumps(profile,indent=2,sort_keys=True)+'\n',encoding='utf-8')
    print('Locked artifact validated; runtime profile:',profile_path,flush=True)
    return {'bundle':bundle,'chunks':chunks,'testset':testset,'sparse_index':sparse_index,'config':config_path,'profile':profile_path}
def run_command(command,label,env_overrides=None):
    log_path=LOGS/f'{label}_{time.strftime("%Y%m%d_%H%M%S")}.log'
    command=[str(value) for value in command]
    print('$',' '.join(command),flush=True); print('Log:',log_path,flush=True)
    with log_path.open('w',encoding='utf-8') as log:
        child_env=os.environ.copy(); child_env.update(env_overrides or {})
        process=subprocess.Popen(command,cwd=PROJECT_ROOT,env=child_env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,encoding='utf-8',errors='replace',bufsize=1)
        for line in process.stdout:
            print(line,end='',flush=True); log.write(line); log.flush()
        returncode=process.wait()
    if returncode:
        write_checkpoint(); raise subprocess.CalledProcessError(returncode,command)
    return log_path
def disk_status():
    usage=shutil.disk_usage(KAGGLE_WORKING)
    value={'free_gib':round(usage.free/2**30,2),'used_gib':round(usage.used/2**30,2),'total_gib':round(usage.total/2**30,2)}
    print('Disk:',value,flush=True); return value


## 2. Lock the Phase 1 retrieval configuration
The retrieval pipeline is locked from the completed Phase 1 protocol: deduplicated resolved questions, BGE-M3 learned sparse retrieval, BGE-large reranking, and recursive chunking 512/64. The selected Round 3 result is embedded below as immutable provenance.

In [ ]:
round3_records=[
    {'index':'chunk_256_32','retrieval.hit_rate@5.mean':0.939502,'retrieval.mrr@5.mean':0.822418,'retrieval.ndcg@5.mean':0.850685,'latency.total.p50_ms':322.6},
    {'index':'chunk_512_64','retrieval.hit_rate@5.mean':0.957295,'retrieval.mrr@5.mean':0.879715,'retrieval.ndcg@5.mean':0.897630,'latency.total.p50_ms':512.7},
    {'index':'chunk_1024_128','retrieval.hit_rate@5.mean':0.925267,'retrieval.mrr@5.mean':0.874673,'retrieval.ndcg@5.mean':0.886183,'latency.total.p50_ms':512.2},
]
eligible=pd.DataFrame(round3_records).sort_values('index')
display(eligible)
selected=eligible[eligible['index']==f'chunk_{LOCKED_CHUNK_SIZE}_{LOCKED_CHUNK_OVERLAP}']
assert len(eligible)==3 and len(selected)==1, 'Embedded Round 3 lock data is incomplete or duplicated'
winner=selected.iloc[0].to_dict(); CHUNK_SIZE,CHUNK_OVERLAP=LOCKED_CHUNK_SIZE,LOCKED_CHUNK_OVERLAP
selection={'source_artifact':'results/phase1/round3.csv','source_scope':'resolved BGE-large rows transcribed after Phase 1 completion','selected_index':winner['index'],'chunk_size':CHUNK_SIZE,'chunk_overlap':CHUNK_OVERLAP,'reranker_model':RERANKER_MODEL,'retriever':'sparse','sparse_model':'BAAI/bge-m3','top_k':TOP_K,'rerank_top_n':RERANK_TOP_N,'selection_metrics':{key:winner[key] for key in ['retrieval.hit_rate@5.mean','retrieval.mrr@5.mean','retrieval.ndcg@5.mean','latency.total.p50_ms']},'round3_candidates':round3_records}
(RESULTS/'retrieval_lock.json').write_text(json.dumps(selection,indent=2,sort_keys=True)+'\n',encoding='utf-8')
print(json.dumps(selection,indent=2))


## 3. API and model preflight
This makes one small request to each locked model through its assigned provider before running the benchmark. No fallback provider or model is allowed.

In [ ]:
from openai import OpenAI
def verify_assigned_model(api_key,model,base_url,output_budget,reasoning_effort):
    client=OpenAI(api_key=api_key,base_url=base_url,timeout=300.0,max_retries=3)
    request={'model':model,'messages':[{'role':'user','content':'Reply with exactly OK'}],'max_tokens':output_budget,'temperature':0.0,'reasoning_effort':reasoning_effort}
    response=client.chat.completions.create(**request)
    assert response.choices, f'No completion choice returned by {model}'
    choice=response.choices[0]; content=choice.message.content
    if not content:
        raise RuntimeError(f'{model} accepted the request but returned no visible content (finish_reason={choice.finish_reason}, usage={response.usage}, max_tokens={output_budget})')
    print('Model access verified:',model)
verify_assigned_model(GENERATOR_API_KEY,GENERATOR_MODEL,'https://generativelanguage.googleapis.com/v1beta/openai/',32,GENERATOR_REASONING_EFFORT)
verify_assigned_model(JUDGE_API_KEY,JUDGE_MODEL,'https://api.fireworks.ai/inference/v1',JUDGE_MAX_TOKENS,JUDGE_REASONING_EFFORT)
disk_status()


## 4. Load the locked corpus and index
The public, immutable Hugging Face artifact contains the reviewed deduplicated `512/64` chunks, resolved testset, and BGE-M3 learned-sparse postings. This cell verifies every bundled file and creates runtime-only config and manifest files for Kaggle. Missing or incompatible files are fatal; Phase 2 does not reconstruct the corpus or index from raw NewsQA.

In [ ]:
locked=validate_and_prepare_locked_index()
chunks_path=locked['chunks']; testset=locked['testset']; profile='bge_m3_bge_large'
spec={'schema_version':1,'experiment':{'id':EXPERIMENT_ID,'name':f'Phase 2 end-to-end baseline ({RUN_MODE})'},'output_dir':str(EXPERIMENTS),'seed':42,'dataset':{'article_field':'article_key','development_articles':DEVELOPMENT_ARTICLES,'indexes':{profile:{'config':str(locked['config']),'variant_manifest':str(locked['profile']),'testsets':{'resolved':str(testset)}}}},'fixed':{'index':profile,'variant':'resolved','partition':'development','retrieval_only':False,'retriever':'sparse','reranker':'cross-encoder','reranker_model':RERANKER_MODEL,'generator_model':GENERATOR_MODEL,'top_k':TOP_K,'rerank_top_n':RERANK_TOP_N},'runtime':{'max_attempts':MAX_ATTEMPTS,'retry_failed':True,'progress':True,'warmup_queries':1,'generation_min_interval_seconds':GENERATOR_MIN_INTERVAL_SECONDS,'shared_retrieval_cache':str(CACHE),**({'n_eval':EVAL_QUESTIONS} if EVAL_QUESTIONS else {})},'judge':{'enabled':False},'pricing':{'provider':'gemini','currency':'USD','input_per_million':0.25,'output_per_million':1.50,'judge':{'provider':'fireworks','model':JUDGE_MODEL,'input_per_million':FIREWORKS_INPUT_PER_MILLION_USD,'output_per_million':FIREWORKS_OUTPUT_PER_MILLION_USD}},'summary':{'metrics':['retrieval.hit_rate@5','retrieval.mrr@5','retrieval.ndcg@5','qa.exact_match','qa.f1','citations.citation_validity','citations.citation_precision','citations.citation_recall','citations.citation_f1','ragas.faithfulness','ragas.answer_relevancy','ragas.context_precision','ragas.context_recall','ragas.answer_correctness'],'paired_metric':'ragas.answer_correctness','quality_metric':'ragas.answer_correctness.mean','latency_metric':'latency.total.p95_ms'}}
spec_path=SPECS/f'{EXPERIMENT_ID}.yaml'; spec_path.write_text(yaml.safe_dump(spec,sort_keys=False),encoding='utf-8')
sys.path.insert(0,str(PROJECT_ROOT/'common'))
from newsqa_rag.experiments import build_article_partitions
partition=build_article_partitions({'resolved':testset},DEVELOPMENT_ARTICLES,42)
development_count=len(partition['partitions']['development']['question_ids']['resolved'])
assert development_count==281, f'Expected 281 development questions, got {development_count}'
assert EVAL_QUESTIONS is None or EVAL_QUESTIONS==5
print('Resolved development pool:',development_count,'| this run:',EVAL_QUESTIONS or development_count); print('Spec:',spec_path); disk_status()


## 5. Collect cited RAG answers
This runs retrieval, reranking, and generation for exactly five seeded questions in smoke mode, or all 281 resolved development questions in full mode. JSONL traces are append-only: rerunning the cell skips successful questions and resumes failed work.

In [ ]:
run_command([sys.executable,'-u','scripts/run_experiment.py',spec_path],'generation',{'GEMINI_API_KEY':GENERATOR_API_KEY})
experiment_dir=EXPERIMENTS/EXPERIMENT_ID
run_dirs=[path.parent for path in experiment_dir.glob('*/experiment_run.json')]
assert len(run_dirs)==1, f'Expected one baseline run, found {len(run_dirs)}'
RUN_DIR=run_dirs[0]
report=json.loads((RUN_DIR/'report.json').read_text()); display(report['coverage']); display(report.get('qa')); display(report.get('citations'))
expected_questions=EVAL_QUESTIONS or 281
assert report['coverage']['expected']==expected_questions, f"Expected {expected_questions} questions, got {report['coverage']['expected']}"
if report['coverage']['success_rate']<0.95: print('WARNING: generation success is below the 95% acceptance target; inspect attempts.jsonl before judging.')
print('Resumable run:',RUN_DIR); write_checkpoint()


## 6. Locked judge validation and RAGAS pilot
The judge configuration is locked to GLM-5.3-Flash with low reasoning and a 2,048-token output limit. Smoke mode validates all five cached answers with this configuration. Full mode first runs the same locked judge on a seeded 25-question pilot.

In [ ]:
JUDGE_METRICS=['faithfulness','answer_relevancy','context_precision','context_recall','answer_correctness']
def load_jsonl(path): return [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]
def judge_command(results_file,n_eval=None,retry=False,require_complete=False):
    command=[sys.executable,'-u','scripts/judge_benchmark_predictions.py','--run-dir',RUN_DIR,'--judge-provider','fireworks','--judge-model',JUDGE_MODEL,'--reasoning-effort',JUDGE_REASONING_EFFORT,'--judge-max-tokens',JUDGE_MAX_TOKENS,'--results-file',results_file,'--batch-size',JUDGE_BATCH_SIZE,'--max-workers',JUDGE_MAX_WORKERS,'--max-attempts',MAX_ATTEMPTS,'--seed',42,'--progress']
    if n_eval: command += ['--n-eval',n_eval]
    if retry: command.append('--retry-failed')
    if require_complete: command.append('--require-complete-metrics')
    return command
def summarize_judge(path):
    records=list({record['question_id']:record for record in load_jsonl(path)}.values()); batches={record.get('batch_id',record['question_id']):record for record in records}; usage={key:sum(record.get('batch_usage',{}).get(key,0) for record in batches.values()) for key in ['input_tokens','output_tokens','total_tokens','successful_requests']}
    row={'model':JUDGE_MODEL,'reasoning_effort':JUDGE_REASONING_EFFORT,'max_tokens':JUDGE_MAX_TOKENS,'questions':len(records),'complete_rows':sum(record.get('status')=='success' for record in records),'partial_rows':sum(record.get('status')=='partial' for record in records),'latency_seconds':round(sum(record.get('batch_elapsed_ms',0) for record in batches.values())/1000,2),**usage}
    row['estimated_cost_usd']=round(usage['input_tokens']/1e6*FIREWORKS_INPUT_PER_MILLION_USD+usage['output_tokens']/1e6*FIREWORKS_OUTPUT_PER_MILLION_USD,6) if usage['total_tokens'] else None
    for metric in JUDGE_METRICS:
        values=[record.get('scores',{}).get(metric) for record in records]; values=[value for value in values if value is not None]; row[f'{metric}_coverage']=round(len(values)/len(records),4) if records else 0; row[f'{metric}_mean']=round(sum(values)/len(values),4) if values else None
    return row
run_command(judge_command('judge_results.jsonl',JUDGE_PILOT_QUESTIONS,require_complete=True),'ragas_pilot',{'FIREWORKS_API_KEY':JUDGE_API_KEY})
judge_summary=pd.DataFrame([summarize_judge(RUN_DIR/'judge_results.jsonl')]); display(judge_summary.T); judge_summary.to_csv(RESULTS/'judge_configuration_summary.csv',index=False)
run_command([sys.executable,'-u','scripts/score_benchmark_predictions.py','--run-dir',RUN_DIR],'score_pilot')
pilot_report=json.loads((RUN_DIR/'report.json').read_text()); display(pilot_report.get('ragas'))
assert pilot_report.get('ragas',{}).get('n_samples',0)==min(JUDGE_PILOT_QUESTIONS,pilot_report['coverage']['successful'])
if RUN_MODE=='smoke': run_command([sys.executable,'-u','scripts/summarize_experiments.py',experiment_dir],'summarize_smoke')
final_report=pilot_report; write_checkpoint()


## 7. Complete RAGAS judging (full mode only)
In smoke mode this cell does not make additional API requests. In full mode, run it after inspecting the 25-question pilot; it reuses completed judgments and evaluates only the remaining successful generations.

In [ ]:
if RUN_MODE=='full':
    run_command(judge_command('judge_results.jsonl',retry=True,require_complete=True),'ragas_full',{'FIREWORKS_API_KEY':JUDGE_API_KEY})
    judge_summary=pd.DataFrame([summarize_judge(RUN_DIR/'judge_results.jsonl')]); display(judge_summary.T); judge_summary.to_csv(RESULTS/'judge_configuration_summary.csv',index=False)
    run_command([sys.executable,'-u','scripts/score_benchmark_predictions.py','--run-dir',RUN_DIR],'score_final')
    run_command([sys.executable,'-u','scripts/summarize_experiments.py',experiment_dir],'summarize')
    final_report=json.loads((RUN_DIR/'report.json').read_text())
else:
    print('Smoke mode: every successful answer from the five-question run was already judged; no additional API calls.')
display(final_report['coverage']); display(final_report.get('ragas'))
ragas_coverage=final_report.get('ragas',{}).get('coverage',0)
if ragas_coverage<0.95: print('WARNING: RAGAS coverage is below 95%; rerun this cell after resolving quota errors.')
write_checkpoint()


## 8. Baseline summary and exports
The table combines deterministic QA/citation metrics, RAGAS means and bootstrap confidence intervals, latency, coverage, token use, and estimated generation cost. Smoke mode also exports the raw five-question traces required for review.

In [ ]:
comparison=pd.read_csv(experiment_dir/'comparison.csv'); display(comparison.T)
score_rows=pd.read_json(RUN_DIR/'deterministic_scores.jsonl',lines=True); scored=pd.json_normalize(score_rows.to_dict('records'))
diagnostics={'pipeline_failures':int((scored['status']!='success').sum()),'retrieval_misses_at_5':int((scored['retrieval.hit_rate@5']==0).sum()),'answers_without_valid_citation':int((scored['citations.citation_validity']==0).sum()),'ragas_rows':int(scored.get('ragas.answer_correctness',pd.Series(dtype=float)).notna().sum())}
display(pd.DataFrame([diagnostics]))
if 'ragas.answer_correctness' in scored: display(scored.nsmallest(10,'ragas.answer_correctness')[['question_id','article_key','qa.f1','citations.citation_f1','ragas.faithfulness','ragas.answer_correctness']])
attempt_path=RUN_DIR/'attempts.jsonl'
if attempt_path.exists():
    attempts=pd.read_json(attempt_path,lines=True); failed_attempts=attempts[attempts['status']=='failed']; display(failed_attempts['stage'].value_counts().rename('failed_attempts'))
metric_rows=[]
for group in ['qa','citations','ragas']:
    for name,value in final_report.get(group,{}).items():
        if isinstance(value,(int,float)) and name not in {'n_samples','coverage'}: metric_rows.append({'group':group,'metric':name,'value':value})
metrics_frame=pd.DataFrame(metric_rows); display(metrics_frame)
import matplotlib.pyplot as plt, seaborn as sns
plt.figure(figsize=(10,5)); sns.barplot(data=metrics_frame,x='metric',y='value',hue='group'); plt.ylim(0,1); plt.xticks(rotation=35,ha='right'); plt.tight_layout(); plt.savefig(RESULTS/'phase2_quality_metrics.png',dpi=180); plt.show(); plt.close()
for name in ['run_manifest.json','environment.json','experiment_run.json','retrievals.jsonl','predictions.jsonl','attempts.jsonl','judge_results.jsonl','deterministic_scores.jsonl','report.json','report_summary.txt']:
    source=RUN_DIR/name
    if source.exists(): shutil.copy2(source,RESULTS/name)
shutil.copy2(experiment_dir/'comparison.csv',RESULTS/'comparison.csv'); shutil.copy2(spec_path,RESULTS/'experiment_spec.yaml')
smoke_manifest={'schema_version':1,'run_mode':RUN_MODE,'expected_questions':expected_questions,'artifact_repo':HF_ARTIFACT_REPO_ID,'artifact_revision':HF_ARTIFACT_REVISION,'artifact_sha256':HF_ARTIFACT_SHA256,'generator_model':GENERATOR_MODEL,'generator_reasoning_effort':GENERATOR_REASONING_EFFORT,'generator_key_secret':'GEMINI_API_KEY_1','judge_provider':'fireworks','judge_model':JUDGE_MODEL,'judge_reasoning_effort':JUDGE_REASONING_EFFORT,'judge_max_tokens':JUDGE_MAX_TOKENS,'judge_key_secret':'FIREWORKS_API_KEY','run_dir':str(RUN_DIR),'generated_at':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime())}
(RESULTS/f'{RUN_MODE}_manifest.json').write_text(json.dumps(smoke_manifest,indent=2,sort_keys=True)+'\n',encoding='utf-8')
archive_name=f'phase2_e2e_baseline_{RUN_MODE}_results'
archive=shutil.make_archive(str(KAGGLE_WORKING/archive_name),'zip',root_dir=RESULTS)
print('Compact results:',archive)
print('Full resumable artifacts:',experiment_dir)
print('Generation usage:',final_report.get('usage')); print('Estimated generation cost:',comparison.get('estimated_generation_cost_usd',pd.Series([None])).iloc[0]); disk_status()
